# RAG With Knowledge Graph using Neo4j

This notebook builds a Retrieval Augmented Generation system that combines a knowledge graph stored in Neo4j with a vector index. The system reads text, extracts entities and relationships with a language model, stores those entities and relationships as a graph, and also stores the text as embeddings for semantic search. When a user asks a question, the system pulls relevant facts from both the graph and the vector index, combines them, and passes that combined context to a language model to produce the final answer.

The notebook is organized below into labeled categories. Each category has a short explanation describing what the code inside it does and why it matters for the overall pipeline. The original code itself is kept exactly as it was written.

## About the LangChain Packages Used

**langchain core**
Contains simple, core abstractions that have emerged as a standard, along with the LangChain Expression Language used to compose these components together.

**langchain community**
Contains third party integrations, including the Neo4j graph and vector store integrations used throughout this notebook.

**langchain**
Contains higher level and use case specific chains, agents, and retrieval algorithms that form the cognitive architecture of the application.

## 1. Installing Required Libraries

Before anything else, the environment needs the packages that power this notebook. `langchain`, `langchain community`, and `langchain openai` provide the chains, integrations, and OpenAI model wrappers. `langchain experimental` provides the `LLMGraphTransformer`, which is the component that turns plain text into graph structured data. `neo4j` is the official Python driver used to talk to the Neo4j database. `wikipedia` lets the notebook pull source articles directly from Wikipedia. `tiktoken` is used internally for token counting when splitting text. `yfiles_jupyter_graphs` renders an interactive visual of the graph directly inside the notebook.

This step only needs to run once per environment.

In [ ]:
%pip install --upgrade --quiet langchain langchain-community langchain-openai langchain-experimental neo4j wikipedia tiktoken yfiles_jupyter_graphs python-dotenv ipywidgets

## 2. Importing Required Libraries

This category gathers every import used later in the notebook. Grouping them together makes it clear, at a glance, which building blocks the pipeline depends on:

1. `RunnableBranch`, `RunnableLambda`, `RunnableParallel`, and `RunnablePassthrough` from `langchain_core.runnables` are the composition primitives used to build the final conversational chain.
2. `ChatPromptTemplate` and `PromptTemplate` are used to construct the prompts sent to the language model.
3. `Tuple`, `List`, and `Optional` from the standard `typing` module are used for type hints, which make the helper functions easier to read and safer to maintain.
4. `AIMessage`, `HumanMessage`, and `StrOutputParser` handle chat message formatting and parsing of the model output into plain text.
5. `ConfigurableField` allows runnable components to expose configurable parameters.
6. `GraphWidget` from `yfiles_jupyter_graphs` and `GraphDatabase` from `neo4j` are used later to visualize the graph interactively.
7. The `os` module is used to store credentials as environment variables so that downstream LangChain integrations can pick them up automatically.

In [1]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [3]:
from typing import Tuple, List, Optional

In [4]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

In [5]:
from langchain_core.runnables import ConfigurableField

In [6]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase


In [7]:
import os
from langchain_neo4j import Neo4jVector

## 3. Setting Up API Keys and Environment Variables

Every external service used in this notebook needs a credential. The OpenAI API key authorizes calls to the language model and the embedding model. The Neo4j URI, username, and password authorize the connection to the graph database instance that will store the extracted knowledge graph.

Since this notebook now runs locally in VS Code instead of Google Colab, credentials are no longer pulled from Colab secret storage with `userdata.get`. Instead, they are loaded from a local `.env` file using `python-dotenv`, then written into `os.environ` so that LangChain integrations such as `Neo4jGraph` and `ChatOpenAI` can read them automatically without needing to pass them explicitly into every function call.

Before running the next cell, create a file named `.env` in the same folder as this notebook with the following content (replace the placeholder values with your own):

```
OPENAI_API_KEY=your-openai-api-key-here
NEO4J_URI=your-neo4j-uri-here
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your-neo4j-password-here
```

Keeping secrets in a `.env` file (and adding `.env` to `.gitignore`) rather than hard coding them inline is what keeps this notebook safe to push to GitHub.

In [9]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

In [17]:
# OPENAI_API_KEY, NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD were already loaded
# into the environment by load_dotenv() above, so no manual assignment is needed here.

## 4. Connecting to the Neo4j Graph Database

With the credentials available as environment variables, `Neo4jGraph` can open a connection to the database instance automatically. This `graph` object is the central handle used throughout the rest of the notebook for writing graph data, running Cypher queries, and creating indexes.

In [10]:
from langchain_community.graphs import Neo4jGraph

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15964\942784274.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.graphs import Neo4jGraph


In [12]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)
driver.verify_connectivity()
print("Connected successfully!")

Connected successfully!


In [13]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

## 5. Loading Source Data from Wikipedia

The knowledge graph needs raw text to learn from. `WikipediaLoader` fetches articles related to the query "Elizabeth I" directly from Wikipedia and returns them as LangChain `Document` objects. Printing the length and previewing the first few documents confirms that the data was retrieved successfully before any further processing happens.

In [14]:
import os

os.environ["USER_AGENT"] = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/138.0 Safari/537.36"
)

In [15]:
from langchain_community.document_loaders import WikipediaLoader

loader = WikipediaLoader(
    query="Elizabeth I",
    load_max_docs=1
)

raw_documents = loader.load()

In [16]:
len(raw_documents)

1

In [17]:
raw_documents[:3]

[Document(metadata={'title': 'Elizabeth I', 'summary': 'Elizabeth I (7 September 1533 – 24 March 1603) was Queen of England and Ireland from 17 November 1558 until her death in 1603. She was the last and longest reigning monarch of the House of Tudor. Her eventful reign, and its effect on history and culture, gave name to the Elizabethan era.\nElizabeth was the only surviving child of Henry VIII and his second wife, Anne Boleyn. When Elizabeth was two years old, her parents\' marriage was annulled, her mother was executed, and Elizabeth was declared illegitimate. Henry restored her to the line of succession when she was 10. After Henry\'s death in 1547, Elizabeth\'s younger half-brother Edward VI ruled until his own death in 1553, bequeathing the crown to a Protestant cousin, Lady Jane Grey, and ignoring the claims of his two half-sisters, Mary and Elizabeth, despite statutes to the contrary. Edward\'s will was quickly set aside and the Catholic Mary became queen, deposing Jane. During

## 6. Splitting Documents into Chunks

Language models have a limited context window, and graph extraction quality tends to degrade on very long passages. `TokenTextSplitter` breaks the raw documents into smaller chunks of five hundred and twelve tokens each, with a twenty four token overlap between consecutive chunks so that context is not lost at chunk boundaries. Only the first three raw documents are used here to keep the example fast and inexpensive to run.

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

documents = splitter.split_documents(raw_documents[:3])

In [19]:
print(len(documents))
print(len(documents[0].page_content))

7
288


## 7. Initializing the Language Model

A single `ChatOpenAI` instance, using the `gpt 3.5 turbo` model with temperature set to zero, is created here and reused for every downstream task that needs a language model, including graph extraction, entity extraction, and final answer generation. A temperature of zero makes the model deterministic, which is desirable for structured extraction tasks where consistency matters more than creativity.

In [20]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [57]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",   # Best Groq model
    api_key=GROQ_API_KEY,
    temperature=0,
)

## 8. Transforming Text into Graph Documents

This is the step where unstructured text becomes structured knowledge. `LLMGraphTransformer` uses the language model defined above to read each chunk of text and identify entities such as people, places, and organizations, along with the relationships that connect them. The result, `graph_documents`, is a list of graph structured representations, one for each input chunk, containing nodes and relationships that can later be written into Neo4j. Printing `graph_documents` lets you inspect exactly what the model extracted before committing it to the database.

In [28]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer = LLMGraphTransformer(
    llm=llm,
    ignore_tool_usage=True
)

In [29]:
graph_documents = []

for i, doc in enumerate(documents):
    try:
        result = llm_transformer.convert_to_graph_documents([doc])
        graph_documents.extend(result)
        print(f"Processed chunk {i + 1}/{len(documents)}")
    except Exception as e:
        print(f"Skipped chunk {i + 1}: {e}")

Processed chunk 1/7
Processed chunk 2/7
Processed chunk 3/7
Processed chunk 4/7
Processed chunk 5/7
Processed chunk 6/7
Skipped chunk 7: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k0epq2vhe5ja70wgp0sdqr8b` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 4583, Requested 4206. Please try again in 5.9175s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


In [ ]:
# graph_documents = llm_transformer.convert_to_graph_documents(documents)

In [30]:
graph_documents

[GraphDocument(nodes=[Node(id='Ireland', type='Country', properties={}), Node(id='Queen of England and Ireland', type='Title', properties={}), Node(id='17 November 1558', type='Date', properties={}), Node(id='7 September 1533', type='Date', properties={}), Node(id='England', type='Country', properties={}), Node(id='Elizabethan era', type='Era', properties={}), Node(id='House of Tudor', type='Organization', properties={}), Node(id='24 March 1603', type='Date', properties={}), Node(id='Elizabeth I', type='Person', properties={})], relationships=[Relationship(source=Node(id='Elizabeth I', type='Person', properties={}), target=Node(id='England', type='Country', properties={}), type='REIGNED_OVER', properties={}), Relationship(source=Node(id='Elizabeth I', type='Person', properties={}), target=Node(id='Ireland', type='Country', properties={}), type='REIGNED_OVER', properties={}), Relationship(source=Node(id='Elizabeth I', type='Person', properties={}), target=Node(id='House of Tudor', type=

## 9. Storing the Graph Documents in Neo4j

The extracted nodes and relationships are written into the Neo4j database using `add_graph_documents`. Setting `baseEntityLabel=True` gives every extracted node a shared base label, which makes it easier to query across different entity types later. Setting `include_source=True` also creates a link back to the original source document for each entity, which preserves traceability between the graph and the text it came from.

In [31]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

## 10. Visualizing the Knowledge Graph

Seeing the graph visually is a useful sanity check before building retrieval on top of it. This category defines a Cypher query that matches nodes and relationships while excluding the internal `MENTIONS` relationship, then uses a small helper function, `showGraph`, to open a direct driver session, run the query, and render the results using an interactive `GraphWidget`. Running `showGraph()` displays the graph inline in the notebook, with node labels mapped to each entity's `id` property for readability.

In [32]:
# directly show the graph resulting from the given Cypher query
default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

In [33]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

In [34]:
def showGraph(cypher: str = default_cypher):
    # create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    display(widget)
    return widget

In [35]:
showGraph()

GraphWidget(layout=Layout(height='800px', width='100%'))

GraphWidget(layout=Layout(height='800px', width='100%'))

## 11. Building a Vector Index for Semantic Search

A knowledge graph is excellent for precise, structured facts, but it does not capture the full nuance of the original text. To cover that gap, this category builds a vector index directly on top of the `Document` nodes already stored in Neo4j from the earlier `include_source=True` step. `Neo4jVector.from_existing_graph` embeds the `text` property of each `Document` node using `OpenAIEmbeddings` and stores the resulting vector in an `embedding` property. Setting `search_type="hybrid"` allows the index to combine vector similarity search with keyword based fulltext search, which tends to produce more robust retrieval than either method alone.

In [36]:
from typing import Tuple, List, Optional

In [40]:
from langchain_neo4j import Neo4jVector

In [43]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [44]:
vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)

## 12. Creating a Fulltext Index for Entity Search

To look up entities in the graph by name quickly and with tolerance for spelling variation, a fulltext index named `entity` is created on the `id` property of every `__Entity__` node. This index powers the structured retrieval step later, where entity names extracted from a user question are matched against nodes already stored in the graph.

In [45]:
graph.query("CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

[]

## 13. Extracting Entities from User Questions

Before the graph can be queried for a user's question, the system needs to know which entities the question is actually about. This category defines an `Entities` schema using Pydantic, describing a list of person, organization, or business names. The `entity_chain` combines a prompt instructing the model to act as an entity extractor with `llm.with_structured_output(Entities)`, which forces the model's response to conform to that schema. The example call shows the chain correctly pulling `"Amelia Earhart"` out of a sample question.

In [48]:
from typing import List
from pydantic import BaseModel, Field

# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        description=(
            "All the person, organization, or business entities "
            "that appear in the text"
        )
    )

In [49]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [60]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

In [61]:
entity_chain = prompt | llm.with_structured_output(Entities)

In [62]:
import json

response = llm.invoke(prompt.invoke({
    "question": "Where was Amelia Earhart born?"
}))

data = json.loads(response.content)
print(data)

{'persons': ['Amelia Earhart'], 'organizations': []}


In [ ]:
# entity_chain.invoke({"question": "Where was Amelia Earhart born?"}).names

## 14. Structured Retrieval Using the Graph

This category performs the actual graph lookup for a question. `remove_lucene_chars` strips characters that would otherwise break a Lucene fulltext query. `generate_full_text_query` turns the extracted entity name into a fuzzy Lucene query, appending a `~2` edit distance tolerance to each word so that minor spelling differences still match. `structured_retriever` ties it together: it calls `entity_chain` to pull entity names out of the question, then for each entity it queries the `entity` fulltext index, finds the matching node, and walks both outgoing and incoming relationships (excluding `MENTIONS`) up to fifty results, formatting each fact as `source relationship target`. The final print statement demonstrates the function pulling structured facts about Elizabeth I directly out of the graph.

In [ ]:
# from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars

In [66]:
import re

LUCENE_SPECIAL_CHARS = r'+-&&||!(){}[]^"~*?:\\/'

def remove_lucene_chars(text: str) -> str:
    pattern = "[" + re.escape(LUCENE_SPECIAL_CHARS) + "]"
    return re.sub(pattern, " ", text)

In [67]:
def generate_full_text_query(input: str) -> str:
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()


In [68]:
# Fulltext index query
def structured_retriever(question: str) -> str:
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [69]:
print(structured_retriever("Who is Elizabeth I?"))

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=104>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 104, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n             

Elizabeth I - REIGNED_OVER -> Ireland
Elizabeth I - REIGNED_OVER -> England
Elizabeth I - MEMBER_OF -> House of Tudor
Elizabeth I - HAS_TITLE -> Queen of England and Ireland
Elizabeth I - REIGN_START -> 17 November 1558
Elizabeth I - REIGN_END -> 24 March 1603
Elizabeth I - BORN_ON -> 7 September 1533
Elizabeth I - DIED_ON -> 24 March 1603
Elizabeth I - GAVE_NAME_TO -> Elizabethan era
Elizabethan era - NAMED_AFTER -> Elizabeth I


## 15. Combining Structured and Unstructured Retrieval

Neither retrieval method alone tells the whole story, so this category merges them. The `retriever` function calls `structured_retriever` to gather graph facts and also calls `vector_index.similarity_search` to gather the most semantically relevant text chunks. Both results are concatenated into a single formatted string labeled `Structured data` and `Unstructured data`, which becomes the context passed to the language model when answering the user's question. This hybrid context is the core idea behind graph augmented retrieval augmented generation.

In [70]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

## 16. Handling Follow up Questions with Chat History

Real conversations often include follow up questions that only make sense with prior context, such as "When was she born?" after previously discussing Elizabeth I. This category defines a condensing prompt, `CONDENSE_QUESTION_PROMPT`, that rewrites a follow up question into a standalone question using the chat history. `_format_chat_history` converts the raw list of human and AI text pairs into proper `HumanMessage` and `AIMessage` objects. Finally, `_search_query` is a `RunnableBranch` that checks whether chat history was actually provided. If it was, the branch condenses the question using the prompt, the language model, and a string output parser. If not, it simply passes the original question straight through unchanged.

In [71]:
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

In [72]:
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

In [73]:
def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

In [76]:
_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | llm
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

## 17. Building the Final RAG Chain

This category assembles every previous piece into one runnable pipeline. The `template` instructs the model to answer strictly using the provided context, in natural and concise language, which helps reduce hallucination. `RunnableParallel` runs two branches at once: the `context` branch sends the (possibly condensed) question through `_search_query` and then through the combined `retriever` to gather graph and vector facts, while the `question` branch simply passes the original input through unchanged. Both outputs feed into the `prompt`, then into the `llm`, and finally through `StrOutputParser` to produce a clean text answer. The result, `chain`, is the complete conversational, graph aware retrieval augmented generation pipeline.

In [77]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

In [78]:
prompt = ChatPromptTemplate.from_template(template)

In [79]:
chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

## 18. Testing the RAG Pipeline

Finally, the complete chain is exercised with two calls. The first call asks a standalone question with no prior history, confirming that the graph and vector retrieval, combined with the language model, produce a correct grounded answer. The second call asks a follow up question, "When was she born?", along with a `chat_history` entry from the earlier turn. This proves that the condensing step correctly rewrites the vague follow up into a standalone question such as "When was Elizabeth I born?" before retrieval happens, so the pipeline can still answer correctly even though the follow up question alone does not mention Elizabeth I by name.

In [80]:
chain.invoke({"question": "Which house did Elizabeth I belong to?"})

Search query: Which house did Elizabeth I belong to?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=104>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 104, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n             

'Elizabeth I belonged to the House of Tudor.'

In [ ]:
chain.invoke(
    {
        "question": "When was Elizabeth I born?",
        "chat_history": [("Which house did Elizabeth I belong to?", "House Of Tudor")],
    }
)

Search query: When was Elizabeth I born?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=104>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 104, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n             

'Elizabeth I was born on\u202f7\u202fSeptember\u202f1533.'

## Conclusion

This notebook demonstrated an end to end pipeline for building a retrieval augmented generation system that is grounded in a knowledge graph rather than plain text chunks alone. Starting from raw Wikipedia articles about Elizabeth I, the pipeline split the text, used a language model to extract entities and relationships, and stored that structured knowledge inside a Neo4j graph database. On top of that same data, a hybrid vector index was built so that semantically relevant passages could still be retrieved even when a fact was not explicitly captured as a graph relationship.

At query time, the system extracts entities from the user's question, traverses the graph around those entities to gather precise structured facts, and simultaneously performs a similarity search to gather supporting unstructured context. These two sources are merged and handed to the language model, which produces a concise, grounded answer. A conversational layer on top of this, built with a question condensing step, allows the system to correctly interpret follow up questions that depend on earlier turns in the conversation.

The overall pattern shown here, combining a knowledge graph with vector search rather than relying on either one alone, tends to produce answers that are both factually precise and contextually rich. This makes it a strong foundation for building question answering systems over structured domains such as biographies, organizational data, product catalogs, or any dataset where the relationships between entities carry as much meaning as the text itself.